# DataSage — Stage 1: Cleaning GRPO Training
Trains Qwen2.5-3B via **Unsloth** + **TRL GRPO** to perform multi-domain data cleaning actions against the DataSage Cleaning OpenEnv environment deployed on HuggingFace Spaces.

**Environment:** [ricalanis/datasage-cleaning](https://huggingface.co/spaces/ricalanis/datasage-cleaning)
**Model:** Qwen2.5-3B-Instruct (4-bit via Unsloth)
**Framework:** TRL GRPOTrainer with environment-in-the-loop rewards

In [ ]:
!pip install -q "openenv-core[core]>=0.2.1"
!pip install -q trl>=0.26 transformers accelerate peft bitsandbytes
!pip install -q vllm
!pip install -q unsloth
!pip install -q wandb

In [ ]:
import json
import os
import re
import random
import torch
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# === Configuration ===
WANDB_PROJECT = "datasage"
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct"
ENV_URL = "https://ricalanis-datasage-cleaning.hf.space"
HF_MODEL_REPO = "ricalanis/datasage-qwen-cleaning"

# Set your API keys
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
# os.environ["HF_TOKEN"] = "your_token_here"
# os.environ["WANDB_API_KEY"] = "your_key_here"

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {BASE_MODEL}")

In [ ]:
SYSTEM_PROMPT = """\
You are a data quality agent. You clean enterprise datasets across multiple \
domains (HR, Sales, Project Management, IT Operations).

Each turn, analyze the data and respond with a JSON cleaning action:
{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}

Available operations:
- fill_null: Fill null values (value can be "median", "mode", or a specific value)
- fix_type: Fix type mismatches in a column (cast to proper type)
- remove_duplicate: Remove duplicate rows
- standardize: Standardize column values (strip whitespace, normalize case)
- trim: Trim whitespace from column values
- correct_typo: Correct typos in categorical values

Think step by step: examine the data quality report, identify the most \
impactful issue, then act."""

TASK_PROMPTS = [
    # HR domain
    "This HR dataset has 23 null values in the MonthlyIncome column. Clean the data.",
    "The HR data has type mismatches in the Age column with values like 'N/A' and '#REF!'. Fix this.",
    "There are duplicate employee records in the HR dataset. Remove them to ensure data integrity.",
    "The Department column has inconsistent casing: 'sales', 'Sales', 'SALES'. Standardize it.",
    "The JobRole column has leading/trailing whitespace in several entries. Trim them.",
    "There are typos in the Attrition column: 'Yse' instead of 'Yes'. Correct the typos.",
    "The YearsAtCompany column has 23 null values and some string entries like 'unknown'. Fix the types first, then fill nulls.",
    "Multiple data quality issues exist in the HR dataset: nulls in MonthlyIncome, type errors in Age, and duplicate rows. Prioritize and fix the most impactful issue first.",
    # Sales domain
    "The Sales data has type mismatches in the Amount column with values like 'N/A' and '#REF!'. Fix this.",
    "There are 15 null values in the Sales Amount column. Fill them with the median.",
    "The Stage column has inconsistent values: 'prospecting', 'Prospecting', 'PROSPECTING'. Standardize.",
    "Duplicate deal records exist in the sales pipeline. Remove duplicates.",
    "The Region column has whitespace issues: '  East  ', ' West'. Trim all values.",
    "The Product column has typos: 'GTX Bsaic' instead of 'GTX Basic'. Correct them.",
    "The DaysInStage column has string values like 'unknown' mixed with numbers. Fix the types.",
    "Multiple quality issues in Sales data: Amount has nulls, Stage has inconsistencies, and there are duplicates. Fix the highest-impact issue.",
    # Project Management domain
    "The PM dataset has 30 nulls in EstimatedHours. Fill with the median value.",
    "The Status column has inconsistent casing: 'in progress', 'In Progress', 'IN PROGRESS'. Standardize.",
    "The ActualHours column has string values like 'N/A' and '#REF!' mixed with numbers. Fix types.",
    "Duplicate task records exist in the project data. Remove them.",
    "The Priority column has leading/trailing whitespace. Trim all values.",
    "The RiskFlag column has typos: 'Hgih' instead of 'High'. Correct them.",
    "The CompletionPct column has type mismatches. Some values are 'TBD' or 'unknown'. Fix this.",
    "Multiple issues in PM data: nulls in EstimatedHours, type errors in ActualHours, and duplicates. Fix the most impactful issue.",
    # IT Operations domain
    "The IT Ops data has type mismatches in the SLATarget column. Values like 'N/A' need fixing.",
    "There are 18 null values in the EscalationLevel column. Fill with the median.",
    "The Category column has inconsistent values: 'hardware', 'Hardware', 'HARDWARE'. Standardize.",
    "Duplicate tickets exist in the IT operations data. Remove duplicates.",
    "The Priority column has whitespace issues: '  P1-Critical  '. Trim values.",
    "The Status column has typos: 'Resovled' instead of 'Resolved'. Correct them.",
    "The ResolutionType column has inconsistent casing. Standardize all values.",
    "Multiple quality issues in IT Ops: SLATarget has type errors, Category is inconsistent, and there are duplicates. Prioritize.",
]


def make_conversation(user_msg):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg},
    ]


dataset = Dataset.from_dict({
    "prompt": [make_conversation(p) for p in TASK_PROMPTS]
})

print(f"Dataset size: {len(dataset)} prompts across 4 domains")

## Reward Functions

Three reward signals drive GRPO training:
1. **Environment Reward** (primary): Steps through the OpenEnv cleaning environment
2. **JSON Format Reward**: Encourages well-structured JSON action output
3. **Reasoning Reward**: Encourages chain-of-thought before acting

In [ ]:
def parse_cleaning_action(text):
    """Parse model output into a cleaning action dict."""
    json_match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if json_match:
        try:
            data = json.loads(json_match.group())
            if "operation" in data and "column" in data:
                return data
        except json.JSONDecodeError:
            pass
    # Fallback: keyword matching
    text_lower = text.lower()
    if "fill" in text_lower or "null" in text_lower:
        return {"operation": "fill_null", "column": "", "value": "median", "params": {}}
    elif "type" in text_lower or "cast" in text_lower:
        return {"operation": "fix_type", "column": "", "value": "numeric", "params": {}}
    elif "duplicate" in text_lower:
        return {"operation": "remove_duplicate", "column": "", "params": {}}
    elif "standard" in text_lower:
        return {"operation": "standardize", "column": "", "params": {}}
    elif "trim" in text_lower:
        return {"operation": "trim", "column": "", "params": {}}
    elif "typo" in text_lower:
        return {"operation": "correct_typo", "column": "", "params": {}}
    return {"operation": "fill_null", "column": "", "value": "median", "params": {}}


def env_reward_fn(completions, **kwargs):
    """Step through the Cleaning environment for each completion. Primary reward."""
    import requests
    rewards = []
    for text in completions:
        try:
            action_dict = parse_cleaning_action(text)
            # Reset environment
            reset_resp = requests.post(f"{ENV_URL}/reset", json={})
            reset_data = reset_resp.json()
            # Step with the parsed action
            step_resp = requests.post(f"{ENV_URL}/step", json={"action": action_dict})
            step_data = step_resp.json()
            rewards.append(float(step_data.get("reward", 0.0)))
        except Exception as e:
            print(f"Env error: {e}")
            rewards.append(0.0)
    return rewards


def json_format_reward(completions, **kwargs):
    """Reward well-formed JSON cleaning actions."""
    rewards = []
    for text in completions:
        if re.search(r'\{[^{}]*"operation"[^{}]*\}', text):
            try:
                match = re.search(r'\{[^{}]*"operation"[^{}]*\}', text)
                data = json.loads(match.group())
                valid_ops = {"fill_null", "fix_type", "remove_duplicate",
                             "standardize", "trim", "correct_typo"}
                if data.get("operation") in valid_ops and "column" in data:
                    rewards.append(1.0)
                elif data.get("operation") in valid_ops:
                    rewards.append(0.6)
                else:
                    rewards.append(0.3)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


def reasoning_reward(completions, **kwargs):
    """Reward chain-of-thought reasoning before the action."""
    rewards = []
    for text in completions:
        score = 0.0
        lower = text.lower()
        if any(w in lower for w in ["first", "let me", "i should", "step 1", "think", "analyze"]):
            score += 0.3
        if any(w in lower for w in ["null", "missing", "type", "duplicate", "whitespace", "typo"]):
            score += 0.2
        if re.search(r'[A-Z][a-z]+(?:[A-Z][a-z]+)+', text):
            score += 0.1
        rewards.append(min(score, 0.5))
    return rewards

## GRPO Training

Using Group Relative Policy Optimization with environment-in-the-loop rewards. Config tuned for Colab T4 GPU (use larger batches on H100).

In [ ]:
training_args = GRPOConfig(
    output_dir="./outputs/cleaning-grpo",
    learning_rate=5e-6,
    num_train_epochs=1,              # 1 epoch for Colab demo (3 on H100)
    per_device_train_batch_size=2,   # 2 for T4 (4 on H100)
    gradient_accumulation_steps=4,
    num_generations=4,               # 4 for T4 (8 on H100)
    max_completion_length=256,       # 256 for T4 (512 on H100)
    max_prompt_length=512,           # 512 for T4 (1024 on H100)
    logging_steps=1,
    save_steps=50,
    bf16=True,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb",
    run_name="datasage-cleaning-grpo",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        env_reward_fn,       # Primary: environment reward
        json_format_reward,  # Auxiliary: structured output
        reasoning_reward,    # Auxiliary: chain-of-thought
    ],
)

print("Starting Stage 1 (Cleaning) GRPO training...")
trainer.train()

In [ ]:
output_dir = "./outputs/cleaning-grpo-final"
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Training complete! Model saved to {output_dir}")

# Push to Hub
trainer.push_to_hub(HF_MODEL_REPO)
print(f"Model pushed to https://huggingface.co/{HF_MODEL_REPO}")